![Notebook 3](images/architecture-detailed.svg)

# 3. Let AgentCore find the bug, then verify its fix

> Deploy the agent, feed it failures on purpose, let AgentCore cluster them and propose a
> prompt rewrite. **Then A/B the fix, because a generated fix is a hypothesis.**

⚠️ This notebook creates billable resources and takes about an hour. The last cell tears
everything down. Notebooks 1 and 2 need neither.

📖 Story and gotchas: [`README.md`](README.md)
💻 Code this notebook imports: [`helpers/`](helpers/) (`agentcore_deploy.py`, `agentcore_loop.py`, `research_checks.py`)
⬅️ Previous: [`02_score_traces_with_agentcore_evaluations.ipynb`](02_score_traces_with_agentcore_evaluations.ipynb)

---

## Prerequisite you cannot skip

**CloudWatch Transaction Search** must be ACTIVE in this region. Without it there are no
spans, and no span means every step below silently scores nothing.

In [ ]:
import json, os, sys, time, uuid
from pathlib import Path
import boto3
from botocore.config import Config as BotoConfig

sys.path.insert(0, "helpers")   # the modules this notebook imports
os.environ.setdefault("AWS_REGION", "us-west-2")
REGION = os.environ["AWS_REGION"]
WORK = Path("agentcore_eval_artifacts"); WORK.mkdir(exist_ok=True)

status = boto3.client("xray", region_name=REGION).get_trace_segment_destination()
print(f"Transaction Search: {status.get('Status')}")
assert status.get("Status") == "ACTIVE", "enable Transaction Search before continuing"

## Deploy

A zip code package, not a container: about a minute instead of 8 to 12, because it skips
CodeBuild and ECR. The entry point is `["opentelemetry-instrument", "main.py"]`, which is
what activates ADOT and therefore what makes the agent evaluable at all.

In [ ]:
from agentcore_deploy import deploy_agent

deployment = deploy_agent(
    name="ResearchAgentEval",
    entrypoint_src=Path("helpers/runtime_entrypoint.py").read_text(),
    modules={"research_agent": Path("helpers/research_agent.py").read_text()},
    deps=["langchain-aws[tools]>=1.7.6", "deepagents>=0.7.13", "bedrock-agentcore>=1.23.0",
          "aws-opentelemetry-distro>=0.18.0", "opentelemetry-instrumentation-langchain>=0.55.0"],
    region=REGION, work_dir=WORK / "build",
)
deployment.save(WORK / "deployment.json")

In [ ]:
dp = boto3.client("bedrock-agentcore", region_name=REGION,
                  # botocore's default 60s read timeout aborts the client while the
                  # agent is still working, which looks like a runtime failure.
                  config=BotoConfig(read_timeout=1200, retries={"max_attempts": 1}))


def invoke(prompt, label, baggage=None):
    """One invocation. Session ids must be >= 33 chars.

    A client timeout is recorded as a failure rather than raised. That is not
    defensive coding: the unguarded prompt used as the A/B control loops until
    LangGraph's recursion ceiling and has been observed running past 900 seconds, so
    "the client gave up" is a real outcome this experiment needs to count.
    """
    sid = f"{label}-{uuid.uuid4().hex}"[:60].ljust(33, "0")
    kw = {"agentRuntimeArn": deployment.runtime_arn, "runtimeSessionId": sid,
          "payload": json.dumps({"prompt": prompt}).encode()}
    if baggage:
        kw["baggage"] = baggage
    t0 = time.time()
    try:
        body = dp.invoke_agent_runtime(**kw)["response"].read().decode()
        out = json.loads(body) if body else {}
    except Exception as exc:
        out = {"result": "", "error": f"{type(exc).__name__}: {exc}"}
    return {**out, "session_id": sid, "elapsed_s": round(time.time() - t0, 1)}

## Feed it failures on purpose

`FailureAnalysis` **clusters failures**. An agent that always succeeds gives the loop
nothing to work with, and `ExecutionSummary` needs at least three sessions. See
`research_checks.py` for what each scenario probes.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from research_checks import FAILURE_SCENARIOS, SUCCESS_SCENARIOS

jobs = SUCCESS_SCENARIOS + FAILURE_SCENARIOS
print(f"sending {len(jobs)} sessions, 4 at a time (4-8 min)\n")

traffic = []
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(invoke, p, l): l for l, p in jobs}
    for f in as_completed(futures):
        r = f.result(); r["label"] = futures[f]
        traffic.append(r)
        print(f"  {r['label']:<26}{r['elapsed_s']:>7.1f}s "
              f"{len(r.get('result','')):>6} chars{'  ERR' if r.get('error') else ''}")
(WORK / "traffic.json").write_text(json.dumps(traffic, indent=2))

## Cluster the failures

Insights and evaluators are **mutually exclusive** in one batch job, and both `aws/spans`
and the runtime log group must be supplied. Takes 15 to 25 minutes.

In [ ]:
from agentcore_loop import clusters, root_causes, run_insights

print("waiting 180s for span ingestion"); time.sleep(180)

detail, INSIGHTS_ARN = run_insights(
    region=REGION, service_name=deployment.service_name,
    log_groups=["aws/spans", deployment.log_group],
)
(WORK / "insights.json").write_text(json.dumps(detail, indent=2, default=str))

In [ ]:
for rc in root_causes(detail):
    print(f"[{rc['fix_type']}] {rc['root_cause']}  ({rc['sessions']} sessions)")
    print(f"    signal: {rc['signal']}")
    print(f"    {rc['explanation'][:180]}\n")
print(clusters(detail))

Root causes tagged `SYSTEM_PROMPT_FIX` are the ones a prompt change can address, which is
exactly what the next step consumes.

## Get a prompt rewrite

We deliberately hand it `LOOPING_COORDINATOR_PROMPT`: the coordinator with its two
filesystem rules removed. That version hunts for the analyst's chart in a filesystem it
cannot see and loops until LangGraph's recursion ceiling.

In [ ]:
from agentcore_loop import recommend_prompt
from research_agent import COMPANIES, COORDINATOR_PROMPT, LOOPING_COORDINATOR_PROMPT

names = ", ".join(c.name for c in COMPANIES)
BROKEN = LOOPING_COORDINATOR_PROMPT.format(company_list=names)
FIXED = COORDINATOR_PROMPT.format(company_list=names)

RECOMMENDED, why = recommend_prompt(
    region=REGION, current_prompt=BROKEN, insights_arn=INSIGHTS_ARN)
print(f"\n{len(BROKEN)} chars -> {len(RECOMMENDED or '')} chars")

In [ ]:
print(why[:1500])

In [ ]:
print(RECOMMENDED)

## Does the recommendation actually address the loop?

Check rather than assume. The answer varies between runs: which failures the insights job
surfaces changes what the rewrite targets.

In [ ]:
signals = ("read_file", "isolated", "working directory", "/tmp", "subagent")
hits = [s for s in signals if RECOMMENDED and s.lower() in RECOMMENDED.lower()]
print(f"filesystem guidance present: {bool(hits)}  {hits}")
print("\nIf empty, this run's recommendation targets something else. That is the point of")
print("the A/B below: a generated fix is a hypothesis, not a result.")

## A/B the fix

Configuration bundles carry a prompt per arm on the **same runtime**, so nothing is
redeployed and the comparison isolates the prompt. Verify each bundle round-trips: one
that does not is a silent no-op, and both arms would run the same prompt.

The control is the broken prompt. The treatment is the known human fix, so this measures
the *effect of the fix itself*. Swap in `RECOMMENDED` to grade the generated one instead.

In [ ]:
from agentcore_loop import make_bundle

S = int(time.time())
control = make_bundle(region=REGION, runtime_arn=deployment.runtime_arn,
                      name=f"Ctl{S}", prompt=BROKEN, note="control: no filesystem rules")
treatment = make_bundle(region=REGION, runtime_arn=deployment.runtime_arn,
                        name=f"Trt{S}", prompt=FIXED, note="treatment: with filesystem rules")
print(f"control {control.verified_chars} chars, treatment {treatment.verified_chars} chars")

In [ ]:
from research_checks import CHART_PROMPT, metrics_found

def run_arm(arm, bundle, n=3):
    with ThreadPoolExecutor(max_workers=3) as pool:
        out = [f.result() for f in as_completed(
            [pool.submit(invoke, CHART_PROMPT, arm, bundle.baggage) for _ in range(n)])]
    for r in out:
        err = r.get("error") or ""
        r["recursion"] = "GraphRecursionError" in err
        r["timeout"] = "ReadTimeout" in err
        r["ok"] = bool(r.get("result"))
        print(f"  {arm:<10}{r['elapsed_s']:>7.1f}s "
              f"{'RECURSION' if r['recursion'] else 'TIMEOUT' if r['timeout'] else ('ok' if r['ok'] else 'ERR')}")
    return out

arm_ctl = run_arm("control", control)
arm_trt = run_arm("treatment", treatment)

## The verdict: measure the thing the fix changes

The built-in judges score answer quality, which this fix barely touches. Trajectory shape
and completion rate are what it changes.

In [ ]:
import statistics
from agentcore_evals import CloudWatchSpanCollector

print("waiting 240s for span ingestion"); time.sleep(240)
collector = CloudWatchSpanCollector(region=REGION)

def summarise(arm, records):
    ok = [r for r in records if r["ok"]]
    trajs = [collector.wait_for_spans(r["session_id"], max_wait_s=300) for r in ok]
    trajs = [t for t in trajs if t.tool_calls]
    return {
        "completed": f"{len(ok)}/{len(records)}",
        "recursion": sum(1 for r in records if r["recursion"]),
        "timeout": sum(1 for r in records if r.get("timeout")),
        "tool_calls": round(statistics.mean(len(t.tool_calls) for t in trajs), 1) if trajs else None,
        "span_s": round(statistics.mean(t.duration_s for t in trajs), 1) if trajs else None,
        "metrics_of_9": round(statistics.mean(metrics_found(r.get("result","")) for r in ok), 1) if ok else None,
        "per_run_calls": [len(t.tool_calls) for t in trajs],
    }

ctl, trt = summarise("control", arm_ctl), summarise("treatment", arm_trt)
print(f"\n  {'metric':<18}{'control':>14}{'treatment':>14}")
for k in ("completed", "recursion", "timeout", "tool_calls", "span_s", "metrics_of_9"):
    print(f"  {k:<18}{str(ctl[k]):>14}{str(trt[k]):>14}")
print(f"\n  per-run tool calls  control {ctl['per_run_calls']}  treatment {trt['per_run_calls']}")
print("  If the arms overlap, it is noise regardless of the means.")

## What to measure with what

| Question | Instrument |
|---|---|
| Is the answer right? | `Correctness`, `Helpfulness`, ground-truth checks |
| Did it achieve the goal? | `GoalSuccessRate` |
| Right tools, right args? | `ToolSelectionAccuracy`, `ToolParameterAccuracy` |
| Expected path? | `TrajectoryInOrderMatch`, leaf tools only |
| Wasting work? | trajectory shape from spans |
| Completing at all? | completion rate and recursion failures |
| Asking before reducing scope? | a custom evaluator; the built-ins penalize asking |

Two things this loop taught us the hard way. The built-in evaluators scored **1.0 in both
arms** of an A/B where one arm used 16% fewer tool calls, so they cannot see wasted work.
And targeting `GoalSuccessRate` made one run's rewrite *add* a confirmation gate and
another's *remove* one, because asking scores as less complete than answering.

## Tear down

Log groups are kept on purpose: they hold the spans every score above references.

In [ ]:
CLEAN = False  # set True and re-run

if not CLEAN:
    print(f"CLEAN is False. Would delete runtime {deployment.runtime_id}, its role,")
    print(f"bundles {control.bundle_id} and {treatment.bundle_id}, and the S3 package.")
    print(f"Kept either way: {deployment.log_group}")
else:
    from agentcore_deploy import delete_deployment
    from agentcore_loop import delete_bundles
    delete_bundles(region=REGION, bundle_ids=[control.bundle_id, treatment.bundle_id])
    delete_deployment(deployment)